# 🎬 Movie Recommendation System
### A complete content-based & collaborative filtering engine

This notebook walks through:
1. **Data Loading & Exploration** — understand our movie dataset
2. **Content-Based Filtering** — recommend based on movie features
3. **Collaborative Filtering** — recommend based on user behaviour
4. **Hybrid Recommender** — combine both for best results
5. **Web App Launch** — serve recommendations via a Flask UI


In [ ]:
# ── Imports ──────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

print("✅ All libraries loaded successfully!")
print(f"   pandas  {pd.__version__}")
print(f"   numpy   {np.__version__}")


In [ ]:
# ── Load Datasets ────────────────────────────────────
movies  = pd.read_csv('data/movies.csv')
ratings = pd.read_csv('data/ratings.csv')

print(f"🎬 Movies  : {movies.shape[0]} rows × {movies.shape[1]} cols")
print(f"⭐ Ratings : {ratings.shape[0]} rows × {ratings.shape[1]} cols")
movies.head(5)


In [ ]:
# ── Exploratory Data Analysis ────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#1a1a2e')
for ax in axes:
    ax.set_facecolor('#16213e')
    ax.tick_params(colors='#e0e0e0')
    for spine in ax.spines.values():
        spine.set_edgecolor('#444')

# 1. Rating distribution
axes[0].hist(movies['rating'], bins=12, color='#e8b4b8', edgecolor='#c0808a', linewidth=1.2)
axes[0].set_title('IMDb Rating Distribution', color='#f5f5f5', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Rating', color='#e0e0e0')
axes[0].set_ylabel('Count', color='#e0e0e0')

# 2. Top 10 genres
all_genres = []
for g in movies['genres']:
    all_genres.extend(g.split('|'))
genre_counts = pd.Series(all_genres).value_counts().head(10)
bars = axes[1].barh(genre_counts.index, genre_counts.values, color='#f0c27f', edgecolor='#c8913a')
axes[1].set_title('Top 10 Genres', color='#f5f5f5', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Count', color='#e0e0e0')
axes[1].invert_yaxis()
for bar, val in zip(bars, genre_counts.values):
    axes[1].text(val + 0.1, bar.get_y() + bar.get_height()/2, str(val), va='center', color='#f5f5f5', fontsize=9)

# 3. Movies per decade
movies['decade'] = (movies['year'] // 10) * 10
decade_counts = movies.groupby('decade').size()
axes[2].bar(decade_counts.index.astype(str), decade_counts.values, color='#a8c9a5', edgecolor='#6a9a67', width=0.6)
axes[2].set_title('Movies per Decade', color='#f5f5f5', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Decade', color='#e0e0e0')
axes[2].set_ylabel('Count', color='#e0e0e0')
axes[2].tick_params(axis='x', rotation=45)

plt.suptitle('Movie Dataset Overview', color='#f5f5f5', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('data/eda_charts.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print("📊 EDA charts saved to data/eda_charts.png")


In [ ]:
# ── Content-Based Filtering ──────────────────────────
# Build a rich feature string per movie
def build_soup(row):
    genres   = row['genres'].replace('|', ' ')
    director = ' '.join(row['director'].split()[:2])          # first 2 words
    cast_kw  = ' '.join(row['cast'].replace(',','').split()[:6])
    desc     = row['description']
    year_tag = f"era_{(row['year']//10)*10}s"
    # Weight genres & director more by repeating
    return f"{genres} {genres} {director} {director} {cast_kw} {desc} {year_tag}"

movies['soup'] = movies.apply(build_soup, axis=1)

tfidf   = TfidfVectorizer(stop_words='english', ngram_range=(1,2))
tfidf_matrix = tfidf.fit_transform(movies['soup'])
cosine_sim   = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Index lookup
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

def content_recommend(title, top_n=5):
    if title not in indices:
        return pd.DataFrame({'Error': [f"'{title}' not found in dataset"]})
    idx   = indices[title]
    sims  = list(enumerate(cosine_sim[idx]))
    sims  = sorted(sims, key=lambda x: x[1], reverse=True)[1:top_n+1]
    movie_indices = [i[0] for i in sims]
    scores        = [round(i[1], 3) for i in sims]
    result = movies.iloc[movie_indices][['title','genres','year','rating']].copy()
    result.insert(0, 'similarity_score', scores)
    return result.reset_index(drop=True)

# Test it
print("🎯 Content-Based Recommendations for 'Inception':")
content_recommend('Inception', top_n=5)


In [ ]:
# ── Collaborative Filtering ──────────────────────────
# Build user-movie matrix
user_movie = ratings.pivot_table(index='userId', columns='movieId', values='rating').fillna(0)

# User-user cosine similarity
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
user_sim_matrix = cos_sim(user_movie)
user_sim_df     = pd.DataFrame(user_sim_matrix,
                                index=user_movie.index,
                                columns=user_movie.index)

def collab_recommend(user_id, top_n=5):
    if user_id not in user_sim_df.index:
        return pd.DataFrame({'Error': [f"User {user_id} not found"]})
    
    similar_users = user_sim_df[user_id].sort_values(ascending=False).iloc[1:6].index.tolist()
    
    # Movies rated by similar users but not by target user
    rated_by_user  = set(ratings[ratings['userId'] == user_id]['movieId'])
    candidate_rows = ratings[ratings['userId'].isin(similar_users)]
    candidates     = candidate_rows[~candidate_rows['movieId'].isin(rated_by_user)]
    
    if candidates.empty:
        return pd.DataFrame({'Message': ['No new recommendations found']})
    
    scores = (candidates.groupby('movieId')['rating']
                        .mean()
                        .sort_values(ascending=False)
                        .head(top_n))
    
    result = movies[movies['movieId'].isin(scores.index)][['movieId','title','genres','year','rating']].copy()
    result = result.merge(scores.rename('predicted_rating').reset_index(), on='movieId')
    result['predicted_rating'] = result['predicted_rating'].round(2)
    return result[['title','genres','year','rating','predicted_rating']].reset_index(drop=True)

print("👥 Collaborative Filtering Recommendations for User 1:")
collab_recommend(user_id=1, top_n=5)


In [ ]:
# ── Hybrid Recommender ───────────────────────────────
def hybrid_recommend(user_id, liked_title, top_n=5, cb_weight=0.5, cf_weight=0.5):
    """
    Blend Content-Based + Collaborative scores.
    cb_weight + cf_weight should sum to 1.
    """
    cb_recs = content_recommend(liked_title, top_n=20)
    if 'Error' in cb_recs.columns:
        return cb_recs
    
    cf_recs = collab_recommend(user_id, top_n=20)
    if 'Error' in cf_recs.columns or 'Message' in cf_recs.columns:
        return cb_recs.head(top_n)
    
    # Normalise each score to 0-1
    scaler = MinMaxScaler()
    cb_recs['cb_norm'] = scaler.fit_transform(cb_recs[['similarity_score']])
    cf_recs['cf_norm'] = scaler.fit_transform(cf_recs[['predicted_rating']])
    
    merged = pd.merge(cb_recs[['title','genres','year','rating','cb_norm']],
                      cf_recs[['title','cf_norm']],
                      on='title', how='outer').fillna(0)
    
    merged['hybrid_score'] = (cb_weight * merged['cb_norm'] +
                               cf_weight * merged['cf_norm'])
    merged = merged.sort_values('hybrid_score', ascending=False).head(top_n)
    merged['hybrid_score'] = merged['hybrid_score'].round(3)
    return merged[['title','genres','year','rating','hybrid_score']].reset_index(drop=True)

print("🔀 Hybrid Recommendations — User 1 liked 'Inception':")
hybrid_recommend(user_id=1, liked_title='Inception', top_n=5)


In [ ]:
# ── Visualise Movie Similarity Heatmap ───────────────
sample_titles = [
    'Inception', 'Interstellar', 'The Matrix', 'Memento', 'Arrival',
    'The Dark Knight', 'The Prestige', 'Parasite', 'Pulp Fiction', 'Fight Club'
]
sample_idx  = [indices[t] for t in sample_titles if t in indices]
sample_sims = cosine_sim[np.ix_(sample_idx, sample_idx)]

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#16213e')

im = ax.imshow(sample_sims, cmap='RdYlGn', vmin=0, vmax=1)
cbar = plt.colorbar(im, ax=ax)
cbar.ax.yaxis.set_tick_params(color='#e0e0e0')
cbar.set_label('Cosine Similarity', color='#e0e0e0', fontsize=11)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='#e0e0e0')

ax.set_xticks(range(len(sample_titles)))
ax.set_yticks(range(len(sample_titles)))
ax.set_xticklabels(sample_titles, rotation=45, ha='right', color='#e0e0e0', fontsize=9)
ax.set_yticklabels(sample_titles, color='#e0e0e0', fontsize=9)
ax.set_title('Movie Similarity Heatmap (Content-Based)', color='#f5f5f5',
             fontsize=14, fontweight='bold', pad=15)

for i in range(len(sample_titles)):
    for j in range(len(sample_titles)):
        val = sample_sims[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                color='black' if val > 0.5 else 'white', fontsize=8)

plt.tight_layout()
plt.savefig('data/similarity_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print("🗺️  Heatmap saved to data/similarity_heatmap.png")


In [ ]:
# ── Save Model Artefacts ─────────────────────────────
import pickle, os

os.makedirs('model', exist_ok=True)

with open('model/cosine_sim.pkl',    'wb') as f: pickle.dump(cosine_sim,   f)
with open('model/movies_df.pkl',     'wb') as f: pickle.dump(movies,       f)
with open('model/user_sim_df.pkl',   'wb') as f: pickle.dump(user_sim_df,  f)
with open('model/ratings_df.pkl',    'wb') as f: pickle.dump(ratings,      f)

print("💾 Model artefacts saved:")
for fn in os.listdir('model'):
    size = os.path.getsize(f'model/{fn}') / 1024
    print(f"   model/{fn}  ({size:.1f} KB)")


In [ ]:
# ── Launch Web App ───────────────────────────────────
# Run this cell LAST — it starts the Flask server
# Then open  http://127.0.0.1:5000  in your browser
# Press the ■ Stop button (or Kernel > Interrupt) to shut it down

import subprocess, sys, threading, time

def run_flask():
    subprocess.run([sys.executable, 'app.py'])

t = threading.Thread(target=run_flask, daemon=True)
t.start()
time.sleep(2)

print("🚀 Flask server started!")
print("👉 Open your browser at:  http://127.0.0.1:5000")
print("\nPress the ■ Stop (Interrupt) button in Jupyter to shut down.")
